# Book Visualizations

Publication-oriented analysis, tables and figures for *Fragrance Accord Intelligence*.

This notebook reproduces the required analytical outputs from the project's source notebooks and prepares them in a consistent format for the book.

## Chapter 1 — Mapping the Fragrance Landscape

### Final Chapter 1 inventory

| Item | Title | Source |
|---|---|---|
| Table 1.1 | Dataset Profile | Dataset profiling |
| Table 1.2 | Availability of Ratings and Accord Data | Dataset profiling |
| Table 1.3 | Fragrance Distribution by Gender Classification | Dataset profiling |
| Table 1.4 | Ten Most Prevalent Fragrance Accords | Notebook 08 |
| Figure 1.1 | Dataset Availability Matrix | Dataset profiling |
| Figure 1.2 | Fragrance Catalogue Distribution by Gender | Dataset profiling |
| Figure 1.3 | Most Prevalent Fragrance Accords | Notebook 08 |

The notebook follows this inventory. Calculations are reproduced for publication and validated before figure production.

**Interpretation conventions**
- A valid accord means a parsed, non-null, non-empty accord label after lower-case and whitespace normalization.
- Accord prevalence uses fragrances with at least one valid accord as its denominator.
- A fragrance may contain multiple accords; prevalence percentages therefore do not sum to 100%.
- Gender labels describe Fragrantica's product classifications, not consumer demographics.
- Ratings and rating counts are platform observations, not sales or direct measures of market demand.

## Chapter 2 — Fragrance Accord Popularity and Ratings

### Chapter 2 inventory

| Item | Title | Source |
|---|---|---|
| Table 2.1 | Overall Accord Prevalence and Ratings | Notebook 08 |
| Table 2.2 | Accord Prevalence by Gender Classification | Notebook 08 |
| Table 2.3 | Gender-Specific Accord Prevalence Gaps | Notebook 08 |
| Table 2.4 | Average Ratings by Gender Classification | Notebook 08 |
| Table 2.5 | Accord Ratings and Community Engagement by Gender | Notebook 08 |
| Figure 2.1 | The Distribution of Accord Prevalence | Notebook 08 |
| Figure 2.2 | Accord Prevalence Across Gender Classifications | Notebook 08 |
| Figure 2.3 | Gender-Specific Accord Prevalence Gaps | Notebook 08 |
| Figure 2.4 | Accord Rating Differences from Gender-Specific Baselines | Notebook 08 |
| Figure 2.5 | Rating Differences and Supporting Evidence | Notebook 08 |

Chapter 2 uses individual normalized accord labels. Accord-pair and accord-triad analyses are reserved for the following chapter.


## 0. Setup and source data

Load the cleaned Fragrantica Delta table and reproduce the accord parsing logic from Notebook 08. The exploded dataset contains one row per fragrance–accord combination.

In [0]:
# Install the two publication-oriented libraries used in this notebook.
# Run this cell once per Databricks cluster/environment.
%pip install lets-plot great-tables

In [0]:
# ============================================================
# NOTEBOOK 14 — BOOK VISUALIZATIONS
# Chapter 1: Mapping the Fragrance Landscape
# ============================================================

from pyspark.sql import functions as F
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import base64
import io

from great_tables import GT, style, loc
from lets_plot import (
    LetsPlot, ggplot, aes, geom_bar, geom_text, coord_flip,
    labs, theme, theme_minimal, element_text, element_line,
    element_rect, element_blank, scale_fill_manual, scale_y_continuous,
    ggsave
)

# Configure Lets-Plot for notebook rendering.
LetsPlot.setup_html()

# Source table
SOURCE_TABLE = "main.fragrance_analytics.fragrantica_perfumes"
df_perfumes = spark.table(SOURCE_TABLE)

# Parse main_accords using the same logic as Notebook 08
df_accords = df_perfumes.withColumn(
    "accords",
    F.from_json(
        F.regexp_replace(
            F.regexp_replace(
                F.col("main_accords"),
                "'",
                '"'
            ),
            r"\bNone\b",
            "null"
        ),
        "array<string>"
    )
)

# One row per fragrance–accord combination.
# Null and empty accord labels are excluded, as in Notebook 08.
df_fragrance_accords = (
    df_accords
    .select(
        "id",
        "name",
        "gender",
        "rating_value",
        "rating_count",
        F.explode("accords").alias("accord")
    )
    .withColumn("accord", F.lower(F.trim(F.col("accord"))))
    .filter(
        F.col("accord").isNotNull()
        & (F.col("accord") != "")
    )
)

print(f"Source table: {SOURCE_TABLE}")
print(f"Source records: {df_perfumes.count():,}")
print(f"Distinct fragrance IDs: {df_perfumes.select('id').distinct().count():,}")
print(f"Distinct valid accord labels: {df_fragrance_accords.select('accord').distinct().count():,}")

## 0.1 Global visualisation standards

One shared editorial design system governs **every table, chart and matrix** in this notebook and is intended to be reused in later chapters.

- **Background:** warm ivory `#F7F3EC` on every surface: figure canvas, plot area, table body, table headers and title area. No dark panels or tinted surfaces.
- **Editorial hierarchy:** a small uppercase bronze section label, a prominent left-aligned serif title, and a concise sans-serif subtitle.
- **Typography:** DejaVu Serif for display titles and key matrix values; DejaVu Sans for table contents, category labels, axes, captions and data labels.
- **Palette:** espresso text `#302824`, muted supporting text `#756B64`, bronze accent `#A47C59`, and subtle ivory-compatible rules/gridlines.
- **Charts:** muted bronze bars, readable value labels, subtle horizontal-axis gridlines, generous whitespace and no unnecessary frames.
- **Tables:** ivory throughout, left-aligned serif titles, restrained subtitles, consistent sans-serif body typography and quiet horizontal rules.
- **Matrices:** ivory quadrants with clear labels, prominent values, percentages and bronze accent markers.
- **Dark-mode protection:** HTML outputs are wrapped in `nodarkmode`; Matplotlib figures are rendered as embedded PNGs with the authored ivory background.

All visualisation cells use the shared constants and helpers below. Do not define local fonts, title sizes or background colours in individual visualisation cells.

In [0]:
# ============================================================
# GLOBAL VISUALISATION DESIGN SYSTEM — ALL TABLES AND CHARTS
# ============================================================

# Single source of truth for colours, typography and sizing.
BOOK_STYLE = {
    "background": "#F7F3EC",
    "surface": "#F7F3EC",
    "text": "#302824",
    "muted": "#756B64",
    "accent": "#A47C59",
    "accent_light": "#D8C3AD",
    "rule": "#E5DDD2",
    "grid": "#E9E2D8",
    "men": "#687F8C",
    "women": "#B47C83",
    "unisex": "#A58B68",
    "other": "#B9B2AC"
}

BOOK_TYPOGRAPHY = {
    "display": "DejaVu Serif",
    "body": "DejaVu Sans"
}

BOOK_FIGURE_CHART_TOP_IN = 1.45  # Fixed physical distance from figure top to chart area

# Shared horizontal-bar layout: chart body starts closer to the left edge.
BOOK_HEADER_X = 0.06
BOOK_BAR_LEFT = 0.15

def apply_book_bar_layout(fig, bottom=0.10, right=0.96):
    """Apply the same left plot margin to every horizontal bar chart."""
    fig.subplots_adjust(
        left=BOOK_BAR_LEFT,
        right=right,
        top=1 - BOOK_FIGURE_CHART_TOP_IN / fig.get_figheight(),
        bottom=bottom
    )

BOOK_SIZES = {
    "title": 22,
    "label": 12,                 # shared readable axis/category/value labels in points
    "bar_value_mm": 4.2,         # Lets-Plot geom_text uses millimetres; approximately 12 pt
    "bar_width": 0.30,           # consistent slim bar thickness across all horizontal charts
    "title_pt": 22,                # Shared figure title size in points
    "table_body_px": "15px",
    "table_title_px": "26px",
    "table_subtitle_px": "15px",
    "section_pt": 10,
    "subtitle_pt": 13,              # Readable shared figure subtitle size in points
    "category_pt": 12,              # Shared category and axis-label size in points
    "value_pt": 12,                 # Match category labels and data-label typography
    "footer_pt": 10,
    "matrix_value_pt": 24
}

plt.rcParams.update({
    "figure.facecolor": BOOK_STYLE["background"],
    "axes.facecolor": BOOK_STYLE["background"],
    "savefig.facecolor": BOOK_STYLE["background"],
    "figure.edgecolor": BOOK_STYLE["background"],
    "text.color": BOOK_STYLE["text"],
    "axes.labelcolor": BOOK_STYLE["text"],
    "xtick.color": BOOK_STYLE["text"],
    "ytick.color": BOOK_STYLE["text"],
    "font.family": BOOK_TYPOGRAPHY["body"],
    "font.size": BOOK_SIZES["label"],
    "axes.titlesize": BOOK_SIZES["title"],
    "axes.titleweight": "bold",
    "axes.titlelocation": "center",
    "axes.labelsize": BOOK_SIZES["label"],
    "savefig.dpi": 300,
    "savefig.bbox": "tight"
})

def make_book_table(data, title, subtitle=None):
    """Build a table using the global ivory, typography and title rules."""
    table = (
        GT(data)
        .tab_header(title=title, subtitle=subtitle)
        .tab_options(
            table_background_color=BOOK_STYLE["background"],
            heading_background_color=BOOK_STYLE["background"],
            column_labels_background_color=BOOK_STYLE["background"],
            table_font_names=[BOOK_TYPOGRAPHY["body"], "Arial", "sans-serif"],
            table_font_size=BOOK_SIZES["table_body_px"],
            table_font_color=BOOK_STYLE["text"],
            heading_align="left",
            heading_title_font_size=BOOK_SIZES["table_title_px"],
            heading_subtitle_font_size=BOOK_SIZES["table_subtitle_px"],
            heading_padding="14px",
            column_labels_font_weight="bold",
            column_labels_padding="10px",
            table_body_hlines_color=BOOK_STYLE["grid"],
            table_body_hlines_width="1px",
            data_row_padding="11px"
        )
    )
    table = table.tab_style(
        style.fill(color=BOOK_STYLE["background"]),
        locations=[loc.body(), loc.column_header(), loc.header()]
    )
    table = table.tab_style(
        style.text(font=BOOK_TYPOGRAPHY["display"], size=BOOK_SIZES["table_title_px"], weight="bold", color=BOOK_STYLE["text"]),
        locations=[loc.title()]
    )
    table = table.tab_style(
        style.text(font=BOOK_TYPOGRAPHY["body"], size=BOOK_SIZES["table_subtitle_px"], color=BOOK_STYLE["muted"]),
        locations=[loc.subtitle()]
    )
    return table

def add_book_figure_header(fig, title, subtitle, section="FRAGRANCE INTELLIGENCE  /  CHAPTER 1"):
    """Apply identical header coordinates, typography and physical spacing to every figure."""
    figure_height = fig.get_figheight()
    section_y = 1 - 0.203 / figure_height
    title_y = 1 - 0.435 / figure_height
    subtitle_y = 1 - 0.841 / figure_height

    fig.text(
        BOOK_HEADER_X, section_y, section,
        color=BOOK_STYLE["accent"],
        fontsize=BOOK_SIZES["section_pt"],
        fontfamily=BOOK_TYPOGRAPHY["body"],
        fontweight="bold",
        ha="left",
        va="top"
    )
    fig.text(
        BOOK_HEADER_X, title_y, title,
        color=BOOK_STYLE["text"],
        fontsize=BOOK_SIZES["title_pt"],
        fontfamily=BOOK_TYPOGRAPHY["display"],
        fontweight="bold",
        ha="left",
        va="top"
    )
    fig.text(
        BOOK_HEADER_X, subtitle_y, subtitle,
        color=BOOK_STYLE["muted"],
        fontsize=BOOK_SIZES["subtitle_pt"],
        fontfamily=BOOK_TYPOGRAPHY["body"],
        ha="left",
        va="top"
    )

def show_book_table(table):
    html = table.as_raw_html(all_important=True)
    displayHTML(f'<div class="nodarkmode">{html}</div>')

def show_book_plot(plot):
    html = plot.to_html(iframe=True)
    displayHTML(f'<div class="nodarkmode">{html}</div>')

def show_book_matplotlib(fig):
    buffer = io.BytesIO()
    # Keep the full figure canvas so titles retain the same visible left position
    # across figures with different tick-label widths.
    fig.savefig(
        buffer, format="png", dpi=240, bbox_inches=None,
        facecolor=BOOK_STYLE["background"],
        edgecolor=BOOK_STYLE["background"]
    )
    image_b64 = base64.b64encode(buffer.getvalue()).decode("ascii")
    displayHTML(
        '<div class="nodarkmode">'
        f'<img src="data:image/png;base64,{image_b64}" '
        'style="display:block;max-width:100%;height:auto;'
        f'background:{BOOK_STYLE["background"]};">'
        '</div>'
    )

BOOK_PLOT_THEME = (
    theme_minimal()
    + theme(
        plot_title=element_text(
            family=BOOK_TYPOGRAPHY["display"],
            size=BOOK_SIZES["title_pt"],
            face="bold",
            color=BOOK_STYLE["text"],
            hjust=0.5
        ),
        plot_subtitle=element_text(
            family=BOOK_TYPOGRAPHY["body"],
            size=BOOK_SIZES["subtitle_pt"],
            color=BOOK_STYLE["muted"],
            hjust=0.5
        ),
        axis_title=element_text(
            family=BOOK_TYPOGRAPHY["body"], size=BOOK_SIZES["label"], color=BOOK_STYLE["muted"]
        ),
        axis_text=element_text(
            family=BOOK_TYPOGRAPHY["body"], size=BOOK_SIZES["label"], color=BOOK_STYLE["text"]
        ),
        axis_text_y=element_text(
            family=BOOK_TYPOGRAPHY["body"], size=BOOK_SIZES["label"], color=BOOK_STYLE["text"]
        ),
        axis_ticks=element_blank(),
        panel_grid_major_y=element_blank(),
        panel_grid_major_x=element_line(color=BOOK_STYLE["grid"], size=0.6),
        panel_grid_minor=element_blank(),
        panel_background=element_rect(fill=BOOK_STYLE["background"], color=BOOK_STYLE["background"]),
        plot_background=element_rect(fill=BOOK_STYLE["background"], color=BOOK_STYLE["background"]),
        legend_position="none"
    )
)

print("Global visualisation standards are ready.")

## 1.1 Table 1.1 — Dataset Profile

The profile distinguishes records with a non-null source accord field from records that produce at least one valid parsed accord. These are not interchangeable measures.

In [0]:
# ============================================================
# TABLE 1.1 — DATASET PROFILE
# ============================================================

total_records = df_perfumes.count()
distinct_ids = df_perfumes.select("id").distinct().count()

valid_accord_ids = (
    df_fragrance_accords
    .select("id")
    .distinct()
)

rated_ids = (
    df_perfumes
    .filter(F.col("rating_value").isNotNull())
    .select("id")
    .distinct()
)

profile = df_perfumes.agg(
    F.count("*").alias("total_records"),
    F.countDistinct("id").alias("distinct_ids"),
    F.countDistinct(
        F.when(F.col("rating_value").isNotNull(), F.col("id"))
    ).alias("records_with_rating"),
    F.countDistinct(
        F.when(F.col("rating_count").isNotNull(), F.col("id"))
    ).alias("records_with_rating_count"),
    F.countDistinct(
        F.when(
            F.col("main_accords").isNotNull(),
            F.col("id")
        )
    ).alias("records_with_non_null_accord_field")
)

valid_accord_count = valid_accord_ids.count()
rated_valid_accord_count = (
    valid_accord_ids
    .join(rated_ids, on="id", how="inner")
    .count()
)

valid_accord_without_rating_count = (
    valid_accord_ids
    .join(rated_ids, on="id", how="left_anti")
    .count()
)

profile_row = profile.first().asDict()

profile_row["records_with_valid_accord_field"] = valid_accord_count
profile_row["rated_records_with_valid_accords"] = rated_valid_accord_count
profile_row["valid_accord_records_without_rating"] = valid_accord_without_rating_count

table_1_1 = pd.DataFrame([
    {"Metric": "Total fragrance records", "Value": profile_row["total_records"]},
    {"Metric": "Distinct fragrance IDs", "Value": profile_row["distinct_ids"]},
    {"Metric": "Records with a rating", "Value": profile_row["records_with_rating"]},
    {"Metric": "Records with a rating count", "Value": profile_row["records_with_rating_count"]},
    {"Metric": "Records with a non-null accord field", "Value": profile_row["records_with_non_null_accord_field"]},
    {"Metric": "Records with at least one valid accord", "Value": profile_row["records_with_valid_accord_field"]},
    {"Metric": "Rated records with at least one valid accord", "Value": profile_row["rated_records_with_valid_accords"]},
    {"Metric": "Valid-accord records without a rating", "Value": profile_row["valid_accord_records_without_rating"]}
])

table_1_1["Value"] = table_1_1["Value"].map(lambda x: f"{x:,}")

gt_table_1_1 = make_book_table(
    table_1_1,
    "Dataset Profile",
    "Coverage and completeness of the Fragrantica catalogue"
)
show_book_table(gt_table_1_1)

## 1.2 Table 1.2 — Availability of Ratings and Accord Data

The four categories below are mutually exclusive. They are calculated from record-level flags rather than inferred by subtracting aggregate totals. The reconciliation assertion verifies that every record belongs to exactly one category.

In [0]:
# ============================================================
# TABLE 1.2 — AVAILABILITY OF RATINGS AND ACCORD DATA
# ============================================================

valid_accord_flag = valid_accord_ids.withColumn(
    "has_valid_accords",
    F.lit(True)
)

availability = (
    df_perfumes
    .select("id", "rating_value")
    .dropDuplicates(["id"])
    .join(valid_accord_flag, on="id", how="left")
    .withColumn(
        "has_rating",
        F.col("rating_value").isNotNull()
    )
    .withColumn(
        "has_valid_accords",
        F.coalesce(F.col("has_valid_accords"), F.lit(False))
    )
)

availability_counts = availability.agg(
    F.count("*").alias("total"),
    F.sum(
        F.when(F.col("has_rating") & F.col("has_valid_accords"), 1).otherwise(0)
    ).alias("accord_and_rating"),
    F.sum(
        F.when(~F.col("has_rating") & F.col("has_valid_accords"), 1).otherwise(0)
    ).alias("accord_only"),
    F.sum(
        F.when(F.col("has_rating") & ~F.col("has_valid_accords"), 1).otherwise(0)
    ).alias("rating_only"),
    F.sum(
        F.when(~F.col("has_rating") & ~F.col("has_valid_accords"), 1).otherwise(0)
    ).alias("neither")
).first().asDict()

group_total = (
    availability_counts["accord_and_rating"]
    + availability_counts["accord_only"]
    + availability_counts["rating_only"]
    + availability_counts["neither"]
)

assert group_total == availability_counts["total"], (
    "Availability groups do not reconcile to the total record count."
)

table_1_2 = pd.DataFrame([
    {
        "Data availability": "Valid accord data and rating",
        "Fragrance records": availability_counts["accord_and_rating"]
    },
    {
        "Data availability": "Valid accord data, no rating",
        "Fragrance records": availability_counts["accord_only"]
    },
    {
        "Data availability": "Rating, no valid accord data",
        "Fragrance records": availability_counts["rating_only"]
    },
    {
        "Data availability": "Neither rating nor valid accord data",
        "Fragrance records": availability_counts["neither"]
    }
])

table_1_2["Share of dataset"] = (
    table_1_2["Fragrance records"] / availability_counts["total"] * 100
).map(lambda x: f"{x:.2f}%")

table_1_2["Fragrance records"] = (
    table_1_2["Fragrance records"].map(lambda x: f"{x:,}")
)

gt_table_1_2 = make_book_table(
    table_1_2,
    "Availability of Ratings and Accord Data",
    "Mutually exclusive record-level availability groups"
)
show_book_table(gt_table_1_2)
print(f"Reconciled records: {group_total:,} / {availability_counts['total']:,}")

## 1.3 Table 1.3 — Fragrance Distribution by Gender Classification

Original category values are retained, including `n/a` and null. Shares use the full distinct-ID catalogue as the denominator. Rating coverage is calculated within each classification.

In [0]:
# ============================================================
# TABLE 1.3 — FRAGRANCE DISTRIBUTION BY GENDER CLASSIFICATION
# ============================================================

gender_source = df_perfumes.withColumn(
    "gender_label",
    F.when(F.col("gender").isNull(), "null")
     .otherwise(F.lower(F.trim(F.col("gender"))))
)

gender_summary = (
    gender_source
    .groupBy("gender_label")
    .agg(
        F.countDistinct("id").alias("fragrance_count"),
        F.countDistinct(
            F.when(F.col("rating_value").isNotNull(), F.col("id"))
        ).alias("rated_fragrance_count"),
        F.countDistinct(
            F.when(F.col("main_accords").isNotNull(), F.col("id"))
        ).alias("records_with_non_null_accord_field")
    )
    .withColumn(
        "share_of_catalogue_pct",
        F.round(F.col("fragrance_count") / F.lit(distinct_ids) * 100, 2)
    )
    .withColumn(
        "rating_coverage_pct",
        F.round(
            F.col("rated_fragrance_count") / F.col("fragrance_count") * 100,
            2
        )
    )
    .orderBy(F.desc("fragrance_count"))
)

table_1_3 = gender_summary.toPandas()

table_1_3 = table_1_3[
    ~table_1_3["gender_label"].isin(["null", "n/a"])
].copy()

table_1_3 = table_1_3.rename(columns={
    "gender_label": "Gender classification",
    "fragrance_count": "Fragrance records",
    "share_of_catalogue_pct": "Share of catalogue",
    "rated_fragrance_count": "Records with rating",
    "rating_coverage_pct": "Rating coverage",
    "records_with_non_null_accord_field": "Records with non-null accord field"
})

table_1_3["Fragrance records"] = table_1_3["Fragrance records"].map(lambda x: f"{x:,}")
table_1_3["Records with rating"] = table_1_3["Records with rating"].map(lambda x: f"{x:,}")
table_1_3["Records with non-null accord field"] = (
    table_1_3["Records with non-null accord field"].map(lambda x: f"{x:,}")
)
table_1_3["Share of catalogue"] = table_1_3["Share of catalogue"].map(lambda x: f"{x:.2f}%")
table_1_3["Rating coverage"] = table_1_3["Rating coverage"].map(lambda x: f"{x:.2f}%")

gt_table_1_3 = make_book_table(
    table_1_3,
    "Fragrance Distribution by Gender Classification",
    "Catalogue representation and availability of rating data"
)
show_book_table(gt_table_1_3)

## 1.4 Table 1.4 — Ten Most Prevalent Fragrance Accords

This reproduces Notebook 08's prevalence definition:

`Prevalence (%) = fragrance count containing accord / fragrances with at least one valid accord × 100`

The rating metrics are included as descriptive context; Chapter 2 will examine rating patterns in greater detail.

In [0]:
# ============================================================
# TABLE 1.4 — TEN MOST PREVALENT FRAGRANCE ACCORDS
# ============================================================

fragrance_accords_unique = df_fragrance_accords.dropDuplicates(["id", "accord"])

accord_prevalence = (
    fragrance_accords_unique
    .groupBy("accord")
    .agg(
        F.countDistinct("id").alias("fragrance_count"),
        F.countDistinct(
            F.when(F.col("rating_value").isNotNull(), F.col("id"))
        ).alias("rated_fragrance_count"),
        F.round(F.avg("rating_value"), 3).alias("avg_rating")
    )
    .withColumn(
        "prevalence_pct",
        F.round(
            F.col("fragrance_count") / F.lit(valid_accord_count) * 100,
            2
        )
    )
)

top_10_accords = (
    accord_prevalence
    .orderBy(F.desc("fragrance_count"), F.asc("accord"))
    .limit(10)
)

table_1_4 = (
    top_10_accords
    .select(
        F.initcap("accord").alias("Accord"),
        F.col("fragrance_count").alias("Fragrance count"),
        F.col("prevalence_pct").alias("Prevalence (%)"),
        F.col("rated_fragrance_count").alias("Rated fragrance count"),
        F.col("avg_rating").alias("Average rating")
    )
    .toPandas()
)

gt_table_1_4 = make_book_table(
    table_1_4,
    "Ten Most Prevalent Fragrance Accords",
    f"Prevalence among {valid_accord_count:,} fragrances with at least one valid accord"
)
show_book_table(gt_table_1_4)
print(f"Prevalence denominator: {valid_accord_count:,} fragrances")

## 1.5 Chapter 1 — Validation checks

Run these checks before using the outputs in the manuscript. The baseline assertions reflect the current dataset snapshot used to prepare Chapter 1. If the source table is intentionally refreshed, review the changed baseline before updating these values.

In [0]:
# ============================================================
# CHAPTER 1 — VALIDATION CHECKS
# ============================================================

assert total_records == 69_948, f"Unexpected record count: {total_records:,}"
assert distinct_ids == total_records, "Fragrance IDs are not unique."
assert valid_accord_count == 68_511, (
    f"Unexpected valid-accord count: {valid_accord_count:,}"
)
assert len(table_1_4) == 10, "Top-10 accord table does not contain 10 rows."

distinct_accord_count = accord_prevalence.select("accord").distinct().count()
assert distinct_accord_count == 90, (
    f"Expected 90 accord labels, found {distinct_accord_count}."
)

print("Chapter 1 table validation passed.")
print(f"Records: {total_records:,}")
print(f"Distinct IDs: {distinct_ids:,}")
print(f"Records with valid accords: {valid_accord_count:,}")
print(f"Distinct accord labels: {distinct_accord_count}")
print(f"Availability groups reconciled: {group_total:,}")

## 1.6 Figure 1.1 — Dataset Availability Matrix

The matrix visualizes the same four mutually exclusive groups as Table 1.2. Counts and percentages are displayed within each cell.

In [0]:
# ============================================================
# FIGURE 1.1 — DATASET AVAILABILITY MATRIX
# Ivory editorial treatment with a shared overline/title/subtitle.
# ============================================================

from matplotlib.patches import Rectangle

fig, ax = plt.subplots(figsize=(12, 7), dpi=180)
fig.patch.set_facecolor(BOOK_STYLE["background"])
ax.set_facecolor(BOOK_STYLE["background"])

matrix = [
    [availability_counts["accord_and_rating"], availability_counts["accord_only"]],
    [availability_counts["rating_only"], availability_counts["neither"]]
]
total = availability_counts["total"]

row_labels = ["Valid accord data", "No valid accord data"]
col_labels = ["Rating available", "Rating unavailable"]
cell_titles = [
    ["Rated + valid accords", "Valid accords only"],
    ["Rated records only", "Neither available"]
]

for row in range(2):
    for col in range(2):
        value = matrix[row][col]
        pct = value / total * 100
        x, y = col, 1 - row

        ax.add_patch(Rectangle(
            (x, y), 1, 1,
            facecolor=BOOK_STYLE["background"],
            edgecolor=BOOK_STYLE["rule"],
            linewidth=1.1,
            zorder=1
        ))
        ax.add_patch(Rectangle(
            (x + 0.075, y + 0.13), 0.035, 0.74,
            facecolor=BOOK_STYLE["accent"],
            edgecolor="none",
            zorder=2
        ))
        ax.text(
            x + 0.54, y + 0.68, cell_titles[row][col],
            ha="center", va="center",
            fontsize=BOOK_SIZES["category_pt"],
            fontfamily=BOOK_TYPOGRAPHY["body"],
            fontweight="bold",
            color=BOOK_STYLE["accent"]
        )
        ax.text(
            x + 0.54, y + 0.45, f"{value:,}",
            ha="center", va="center",
            fontsize=BOOK_SIZES["matrix_value_pt"],
            fontfamily=BOOK_TYPOGRAPHY["display"],
            fontweight="bold",
            color=BOOK_STYLE["text"]
        )
        ax.text(
            x + 0.54, y + 0.24, f"{pct:.1f}% of catalogue",
            ha="center", va="center",
            fontsize=BOOK_SIZES["value_pt"],
            fontfamily=BOOK_TYPOGRAPHY["body"],
            color=BOOK_STYLE["muted"]
        )

ax.set_xlim(0, 2)
ax.set_ylim(0, 2)
ax.set_xticks([0.5, 1.5], col_labels)
ax.set_yticks([1.5, 0.5], row_labels)
ax.tick_params(length=0, labelsize=BOOK_SIZES["category_pt"], colors=BOOK_STYLE["text"], pad=13)

for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontfamily(BOOK_TYPOGRAPHY["body"])
    tick.set_fontsize(BOOK_SIZES["category_pt"])
    tick.set_color(BOOK_STYLE["text"])

for spine in ax.spines.values():
    spine.set_visible(False)

add_book_figure_header(
    fig,
    "Dataset Availability Matrix",
    "Coverage of rating and valid accord data across the full fragrance catalogue."
)
fig.text(
    0.5, 0.035,
    f"Total dataset: {total:,} fragrance records. Each quadrant reports count and share of the catalogue.",
    ha="center",
    fontsize=BOOK_SIZES["footer_pt"],
    fontfamily=BOOK_TYPOGRAPHY["body"],
    color=BOOK_STYLE["muted"]
)

plt.subplots_adjust(left=0.16, right=0.96, top=1 - BOOK_FIGURE_CHART_TOP_IN / fig.get_figheight(), bottom=0.13)
show_book_matplotlib(fig)

## 1.7 Figure 1.2 — Fragrance Catalogue Distribution by Gender

The three principal Fragrantica classifications are shown as horizontal bars. The number of records outside these three categories is noted separately so that the very small `n/a` and null categories remain visible without distorting the chart scale.

In [0]:
# ============================================================
# FIGURE 1.2 — FRAGRANCE CATALOGUE DISTRIBUTION BY GENDER
# Ivory editorial treatment; labels show count and catalogue share.
# ============================================================

gender_plot = gender_summary.toPandas()

main_categories = gender_plot[
    gender_plot["gender_label"].isin(["men", "women", "unisex"])
].copy()

other_count = int(
    gender_plot[
        ~gender_plot["gender_label"].isin(["men", "women", "unisex"])
    ]["fragrance_count"].sum()
)

main_categories["category_label"] = main_categories["gender_label"].str.title()
main_categories = main_categories.sort_values("fragrance_count", ascending=True)
main_categories["display_label"] = main_categories.apply(
    lambda row: f"{int(row['fragrance_count']):,}  |  {row['fragrance_count'] / distinct_ids * 100:.2f}%",
    axis=1
)

fig_1_2, ax = plt.subplots(figsize=(12, 5.8), dpi=180)
fig_1_2.patch.set_facecolor(BOOK_STYLE["background"])
ax.set_facecolor(BOOK_STYLE["background"])

bars = ax.barh(
    main_categories["category_label"],
    main_categories["fragrance_count"],
    height=0.62,
    color=BOOK_STYLE["accent"],
    edgecolor="none",
    zorder=3
)

max_value = float(main_categories["fragrance_count"].max())
ax.set_xlim(0, max_value * 1.36)

for bar, label in zip(bars, main_categories["display_label"]):
    ax.text(
        bar.get_width() + max_value * 0.022,
        bar.get_y() + bar.get_height() / 2,
        label,
        va="center",
        ha="left",
        fontsize=BOOK_SIZES["value_pt"],
        fontfamily=BOOK_TYPOGRAPHY["body"],
        color=BOOK_STYLE["text"]
    )

add_book_figure_header(
    fig_1_2,
    "Fragrance Catalogue Distribution by Gender",
    "Fragrantica product classifications. Bar-end labels show fragrance count  |  % of the full catalogue."
)

ax.set_xlabel("")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
ax.tick_params(axis="y", length=0, labelsize=BOOK_SIZES["category_pt"], pad=13, colors=BOOK_STYLE["text"])
ax.tick_params(axis="x", length=0, labelsize=BOOK_SIZES["footer_pt"], pad=9, colors=BOOK_STYLE["muted"])

for tick in ax.get_yticklabels():
    tick.set_fontfamily(BOOK_TYPOGRAPHY["body"])
    tick.set_fontsize(BOOK_SIZES["category_pt"])
    tick.set_color(BOOK_STYLE["text"])

for tick in ax.get_xticklabels():
    tick.set_fontfamily(BOOK_TYPOGRAPHY["body"])

ax.grid(
    axis="x",
    color=BOOK_STYLE["grid"],
    linewidth=0.8,
    alpha=0.9,
    zorder=0
)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_visible(False)

fig_1_2.text(
    0.5, 0.035,
    f"Other / unclassified categories: {other_count:,} records (including n/a and null). Shares use the full catalogue as denominator.",
    ha="center",
    fontsize=BOOK_SIZES["footer_pt"],
    fontfamily=BOOK_TYPOGRAPHY["body"],
    color=BOOK_STYLE["muted"]
)

apply_book_bar_layout(fig_1_2, bottom=0.16, right=0.85)
show_book_matplotlib(fig_1_2)

## 1.8 Figure 1.3 — Most Prevalent Fragrance Accords

The figure uses the same top-ten ranking as Table 1.4. Each bar represents the number of fragrances containing that accord; labels provide both count and prevalence. The denominator is the number of fragrances with at least one valid accord.

In [0]:
# ============================================================
# FIGURE 1.3 — MOST PREVALENT FRAGRANCE ACCORDS
# Ivory editorial treatment; labels show count and valid-accord share.
# ============================================================

accord_plot = top_10_accords.toPandas().sort_values(
    "fragrance_count",
    ascending=True
).copy()

accord_plot["accord_label"] = accord_plot["accord"].str.title()
accord_plot["display_label"] = accord_plot.apply(
    lambda row: f"{int(row['fragrance_count']):,}  |  {row['prevalence_pct']:.2f}%",
    axis=1
)

fig_1_3, ax = plt.subplots(figsize=(12, 8.5), dpi=180)
fig_1_3.patch.set_facecolor(BOOK_STYLE["background"])
ax.set_facecolor(BOOK_STYLE["background"])

bars = ax.barh(
    accord_plot["accord_label"],
    accord_plot["fragrance_count"],
    height=0.62,
    color=BOOK_STYLE["accent"],
    edgecolor="none",
    zorder=3
)

max_value = float(accord_plot["fragrance_count"].max())
ax.set_xlim(0, max_value * 1.36)

for bar, label in zip(bars, accord_plot["display_label"]):
    ax.text(
        bar.get_width() + max_value * 0.022,
        bar.get_y() + bar.get_height() / 2,
        label,
        va="center",
        ha="left",
        fontsize=BOOK_SIZES["value_pt"],
        fontfamily=BOOK_TYPOGRAPHY["body"],
        color=BOOK_STYLE["text"]
    )

add_book_figure_header(
    fig_1_3,
    "Ten Most Prevalent Fragrance Accords",
    "Bar-end labels show fragrance count  |  % of fragrances with at least one valid accord."
)

ax.set_xlabel("")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
ax.tick_params(axis="y", length=0, labelsize=BOOK_SIZES["category_pt"], pad=13, colors=BOOK_STYLE["text"])
ax.tick_params(axis="x", length=0, labelsize=BOOK_SIZES["footer_pt"], pad=9, colors=BOOK_STYLE["muted"])

for tick in ax.get_yticklabels():
    tick.set_fontfamily(BOOK_TYPOGRAPHY["body"])
    tick.set_fontsize(BOOK_SIZES["category_pt"])
    tick.set_color(BOOK_STYLE["text"])

for tick in ax.get_xticklabels():
    tick.set_fontfamily(BOOK_TYPOGRAPHY["body"])

ax.grid(
    axis="x",
    color=BOOK_STYLE["grid"],
    linewidth=0.8,
    alpha=0.9,
    zorder=0
)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_visible(False)

fig_1_3.text(
    0.5, 0.025,
    f"Prevalence denominator: {valid_accord_count:,} fragrances with at least one valid accord. Accords may overlap within a fragrance.",
    ha="center",
    fontsize=BOOK_SIZES["footer_pt"],
    fontfamily=BOOK_TYPOGRAPHY["body"],
    color=BOOK_STYLE["muted"]
)

apply_book_bar_layout(fig_1_3, bottom=0.10, right=0.85)
show_book_matplotlib(fig_1_3)

## 1.9 Optional asset export

After reviewing the rendered prototypes, set `EXPORT_BOOK_ASSETS = True` to save the figures as high-resolution PNG and scalable SVG files. Great Tables can also export the formatted tables as HTML; PNG/PDF table export requires a compatible headless Chrome/Chromium runtime, so it is deliberately kept optional.

In [0]:
# ============================================================
# OPTIONAL EXPORT — RUN ONLY AFTER VISUAL REVIEW
# ============================================================

from pathlib import Path

EXPORT_BOOK_ASSETS = False
BOOK_EXPORT_DIR = Path("/dbfs/FileStore/fragrance_book/chapter_1")

if EXPORT_BOOK_ASSETS:
    BOOK_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

    # Export all three Matplotlib figures as high-resolution PNG and SVG.
    for figure, stem in [
        (fig, "figure_1_1_dataset_availability"),
        (fig_1_2, "figure_1_2_gender_distribution"),
        (fig_1_3, "figure_1_3_prevalent_accords")
    ]:
        figure.savefig(
            BOOK_EXPORT_DIR / f"{stem}.png",
            dpi=400,
            bbox_inches="tight",
            facecolor=BOOK_STYLE["background"]
        )
        figure.savefig(
            BOOK_EXPORT_DIR / f"{stem}.svg",
            bbox_inches="tight",
            facecolor=BOOK_STYLE["background"]
        )

    # Keep styled tables available as portable HTML files.
    for table_name, table_object in {
        "table_1_1_dataset_profile": gt_table_1_1,
        "table_1_2_data_availability": gt_table_1_2,
        "table_1_3_gender_distribution": gt_table_1_3,
        "table_1_4_prevalent_accords": gt_table_1_4
    }.items():
        table_object.write_raw_html(
            str(BOOK_EXPORT_DIR / f"{table_name}.html")
        )

    print(f"Book assets exported to: {BOOK_EXPORT_DIR}")
else:
    print("Asset export is off. Review the notebook-rendered prototypes first.")

# Chapter 2 — Fragrance Accord Popularity and Ratings

This section reproduces the individual-accord prevalence, gender comparison and rating analyses from Notebook 08. It does not use dominant-accord, pair or triad metrics.

The unit of analysis is a fragrance–accord association. A fragrance may contribute to more than one accord group. Prevalence percentages are therefore not mutually exclusive.

## 2.0 Chapter 2 — Analytical data preparation

The following cell recreates the measures used in Notebook 08 from the same cleaned Fragrantica source table. It prepares the overall prevalence, gender-specific prevalence, rating and engagement measures, and gender-specific rating baselines.

In [0]:
# ============================================================
# CHAPTER 2 — ANALYTICAL DATA PREPARATION
# Source methodology: Notebook 08
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

MIN_RATED_FRAGRANCES = 100
CH2_GENDERS = ["men", "unisex", "women"]

# One record per fragrance, retaining the fields needed below.
fragrance_ratings_ch2 = (
    df_perfumes
    .select("id", "gender", "rating_value", "rating_count")
    .dropDuplicates(["id"])
    .withColumn("gender", F.lower(F.trim(F.col("gender"))))
)

# One row per unique fragrance–accord association.
fragrance_accords_ch2 = (
    df_fragrance_accords
    .select("id", "gender", "accord")
    .withColumn("gender", F.lower(F.trim(F.col("gender"))))
    .dropDuplicates(["id", "accord"])
)

valid_accord_count_ch2 = fragrance_accords_ch2.select("id").distinct().count()
distinct_accord_count_ch2 = fragrance_accords_ch2.select("accord").distinct().count()

# Overall prevalence and rating coverage by accord.
accord_prevalence_ch2 = (
    fragrance_accords_ch2
    .groupBy("accord")
    .agg(F.countDistinct("id").alias("fragrance_count"))
    .withColumn(
        "prevalence_pct",
        F.round(F.col("fragrance_count") / F.lit(valid_accord_count_ch2) * 100, 2)
    )
)

accord_rating_overall_ch2 = (
    fragrance_accords_ch2.alias("a")
    .join(
        fragrance_ratings_ch2.select("id", "rating_value", "rating_count").alias("r"),
        on="id",
        how="left"
    )
    .groupBy("accord")
    .agg(
        F.countDistinct(
            F.when(F.col("rating_value").isNotNull(), F.col("a.id"))
        ).alias("rated_fragrance_count"),
        F.round(F.avg("rating_value"), 3).alias("avg_rating"),
        F.percentile_approx("rating_value", 0.5, 10000).alias("median_rating"),
        F.sum("rating_count").alias("total_rating_count"),
        F.round(F.avg("rating_count"), 1).alias("avg_rating_count")
    )
)

accord_prevalence_ch2 = (
    accord_prevalence_ch2
    .join(accord_rating_overall_ch2, on="accord", how="left")
    .orderBy(F.desc("fragrance_count"), F.asc("accord"))
)

# Gender-specific prevalence: each category has its own denominator.
gender_denominators_ch2 = (
    fragrance_accords_ch2
    .filter(F.col("gender").isin(CH2_GENDERS))
    .select("id", "gender").distinct()
    .groupBy("gender")
    .agg(F.countDistinct("id").alias("gender_fragrance_count"))
)

accord_gender_prevalence_ch2 = (
    fragrance_accords_ch2
    .filter(F.col("gender").isin(CH2_GENDERS))
    .groupBy("gender", "accord")
    .agg(F.countDistinct("id").alias("fragrance_count"))
    .join(gender_denominators_ch2, on="gender", how="inner")
    .withColumn(
        "prevalence_pct",
        F.round(F.col("fragrance_count") / F.col("gender_fragrance_count") * 100, 2)
    )
)

# Pivot category-specific prevalence to one row per accord.
accord_gender_wide_ch2 = (
    accord_gender_prevalence_ch2
    .groupBy("accord")
    .pivot("gender", CH2_GENDERS)
    .agg(F.max("prevalence_pct"))
    .fillna(0.0)
)

accord_gender_gaps_ch2 = (
    accord_gender_wide_ch2
    .withColumn(
        "max_prevalence_pct",
        F.greatest("men", "unisex", "women")
    )
    .withColumn(
        "min_prevalence_pct",
        F.least("men", "unisex", "women")
    )
    .withColumn(
        "gap_pp",
        F.round(F.col("max_prevalence_pct") - F.col("min_prevalence_pct"), 2)
    )
    .join(
        accord_prevalence_ch2.select("accord", "fragrance_count", "prevalence_pct"),
        on="accord",
        how="left"
    )
)

common_accord_gaps_ch2 = (
    accord_gender_gaps_ch2
    .filter(F.col("prevalence_pct") >= 10)
    .orderBy(F.desc("gap_pp"), F.asc("accord"))
)

# Gender-specific ratings and community engagement by accord.
accord_rating_by_gender_ch2 = (
    fragrance_accords_ch2
    .filter(F.col("gender").isin(CH2_GENDERS))
    .join(
        fragrance_ratings_ch2.select("id", "rating_value", "rating_count"),
        on="id",
        how="left"
    )
    .groupBy("gender", "accord")
    .agg(
        F.countDistinct("id").alias("fragrance_count"),
        F.count("rating_value").alias("rated_fragrance_count"),
        F.round(F.avg("rating_value"), 3).alias("avg_rating"),
        F.round(F.percentile_approx("rating_value", 0.5, 10000), 3).alias("median_rating"),
        F.sum("rating_count").alias("total_rating_count"),
        F.round(F.avg("rating_count"), 1).alias("avg_rating_count")
    )
)

gender_rating_baseline_ch2 = (
    fragrance_ratings_ch2
    .filter(F.col("gender").isin(CH2_GENDERS))
    .groupBy("gender")
    .agg(
        F.count("rating_value").alias("baseline_rated_count"),
        F.round(F.avg("rating_value"), 3).alias("gender_avg_rating"),
        F.round(F.percentile_approx("rating_value", 0.5, 10000), 3).alias("gender_median_rating")
    )
)

accord_rating_analysis_ch2 = (
    accord_rating_by_gender_ch2
    .filter(F.col("rated_fragrance_count") >= MIN_RATED_FRAGRANCES)
    .join(gender_rating_baseline_ch2, on="gender", how="inner")
    .withColumn(
        "rating_difference",
        F.round(F.col("avg_rating") - F.col("gender_avg_rating"), 3)
    )
)

print(f"Valid-accord fragrance denominator: {valid_accord_count_ch2:,}")
print(f"Distinct normalized accord labels: {distinct_accord_count_ch2:,}")
print(f"Minimum rated-fragrance group size for rating comparisons: {MIN_RATED_FRAGRANCES}")
print("Gender categories included: Men, Unisex and Women")

## 2.1 Table 2.1 — Overall Accord Prevalence and Ratings

All normalized accord labels are retained. Prevalence uses the valid-accord fragrance denominator; rating measures are descriptive and use the fragrances associated with each accord.

In [0]:
# ============================================================
# TABLE 2.1 — OVERALL ACCORD PREVALENCE AND RATINGS
# ============================================================

table_2_1 = (
    accord_prevalence_ch2
    .select(
        F.initcap("accord").alias("Accord"),
        F.col("fragrance_count").alias("Fragrance count"),
        F.col("prevalence_pct").alias("Prevalence (%)"),
        F.col("rated_fragrance_count").alias("Rated fragrance count"),
        F.col("avg_rating").alias("Average rating"),
        F.col("median_rating").alias("Median rating"),
        F.col("total_rating_count").alias("Total community ratings"),
        F.col("avg_rating_count").alias("Average ratings per fragrance")
    )
    .toPandas()
)
gt_table_2_1 = make_book_table(
    table_2_1,
    "Overall Accord Prevalence and Ratings",
    f"All {distinct_accord_count_ch2} normalized accords; prevalence denominator: {valid_accord_count_ch2:,} fragrances"
)
show_book_table(gt_table_2_1)

## 2.2 Table 2.2 — Accord Prevalence by Gender Classification

Prevalence is calculated independently within each Fragrantica category. The percentages are not shares of the combined catalogue.

In [0]:
# ============================================================
# TABLE 2.2 — ACCORD PREVALENCE BY GENDER CLASSIFICATION
# ============================================================

table_2_2 = (
    accord_gender_prevalence_ch2
    .groupBy("accord")
    .pivot("gender", CH2_GENDERS)
    .agg(F.max("prevalence_pct"))
    .fillna(0.0)
    .join(
        accord_prevalence_ch2.select("accord", "fragrance_count", "prevalence_pct"),
        on="accord",
        how="left"
    )
    .select(
        F.initcap("accord").alias("Accord"),
        F.col("fragrance_count").alias("Overall fragrance count"),
        F.col("prevalence_pct").alias("Overall prevalence (%)"),
        F.col("men").alias("Men (%)"),
        F.col("unisex").alias("Unisex (%)"),
        F.col("women").alias("Women (%)")
    )
    .orderBy(F.desc("Overall prevalence (%)"))
    .toPandas()
)
gt_table_2_2 = make_book_table(
    table_2_2,
    "Accord Prevalence by Gender Classification",
    "Within-category prevalence; each gender column uses its own fragrance denominator"
)
show_book_table(gt_table_2_2)

## 2.3 Table 2.3 — Gender-Specific Accord Prevalence Gaps

The table is restricted to accords with overall prevalence of at least 10%. The gap is the highest minus the lowest prevalence across Men, Unisex and Women; it is not necessarily a direct Men–Women comparison.

In [0]:
# ============================================================
# TABLE 2.3 — GENDER-SPECIFIC ACCORD PREVALENCE GAPS
# ============================================================

table_2_3 = (
    common_accord_gaps_ch2
    .select(
        F.initcap("accord").alias("Accord"),
        F.col("fragrance_count").alias("Overall fragrance count"),
        F.col("prevalence_pct").alias("Overall prevalence (%)"),
        F.col("men").alias("Men (%)"),
        F.col("unisex").alias("Unisex (%)"),
        F.col("women").alias("Women (%)"),
        F.col("gap_pp").alias("Prevalence gap (pp)")
    )
    .toPandas()
)
gt_table_2_3 = make_book_table(
    table_2_3,
    "Gender-Specific Accord Prevalence Gaps",
    "Accords with at least 10% overall prevalence; gap = highest minus lowest category prevalence"
)
show_book_table(gt_table_2_3)

## 2.4 Table 2.4 — Average Ratings by Gender Classification

These category averages provide the reference baselines used in the accord-level comparisons.

In [0]:
# ============================================================
# TABLE 2.4 — AVERAGE RATINGS BY GENDER CLASSIFICATION
# ============================================================

table_2_4 = (
    gender_rating_baseline_ch2
    .select(
        F.initcap("gender").alias("Fragrantica category"),
        F.col("baseline_rated_count").alias("Rated fragrance count"),
        F.col("gender_avg_rating").alias("Average rating"),
        F.col("gender_median_rating").alias("Median rating")
    )
    .orderBy("Fragrantica category")
    .toPandas()
)
gt_table_2_4 = make_book_table(
    table_2_4,
    "Average Ratings by Gender Classification",
    "Category-level baselines calculated from rated fragrances in each Fragrantica classification"
)
show_book_table(gt_table_2_4)

## 2.5 Table 2.5 — Accord Ratings and Community Engagement by Gender

The table retains the number of fragrances, number of rated fragrances, average and median ratings, and community rating counts. Rating-difference comparisons are limited to groups with at least 100 rated fragrances.

In [0]:
# ============================================================
# TABLE 2.5 — ACCORD RATINGS AND COMMUNITY ENGAGEMENT BY GENDER
# ============================================================

table_2_5 = (
    accord_rating_analysis_ch2
    .select(
        F.initcap("gender").alias("Fragrantica category"),
        F.initcap("accord").alias("Accord"),
        F.col("fragrance_count").alias("Fragrance count"),
        F.col("rated_fragrance_count").alias("Rated fragrance count"),
        F.col("avg_rating").alias("Average rating"),
        F.col("median_rating").alias("Median rating"),
        F.col("gender_avg_rating").alias("Category baseline"),
        F.col("rating_difference").alias("Difference from baseline"),
        F.col("total_rating_count").alias("Total community ratings"),
        F.col("avg_rating_count").alias("Average ratings per fragrance")
    )
    .orderBy("Fragrantica category", F.desc("Difference from baseline"), "Accord")
    .toPandas()
)
gt_table_2_5 = make_book_table(
    table_2_5,
    "Accord Ratings and Community Engagement by Gender",
    f"Accord/category groups with at least {MIN_RATED_FRAGRANCES} rated fragrances"
)
show_book_table(gt_table_2_5)

## 2.6 Figure 2.1 — The Distribution of Accord Prevalence

The figure displays the 30 most prevalent accords for legibility. Table 2.1 retains all 90 normalized labels. Prevalence is the percentage of valid-accord fragrances containing each accord; accords may overlap.

In [0]:
# ============================================================
# FIGURE 2.1 — THE DISTRIBUTION OF ACCORD PREVALENCE
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

pdf_2_1 = (
    accord_prevalence_ch2
    .orderBy(F.desc("prevalence_pct"), F.asc("accord"))
    .limit(30)
    .toPandas()
    .sort_values("prevalence_pct", ascending=True)
)

fig_2_1, ax = plt.subplots(figsize=(12, 10), dpi=180)
fig_2_1.patch.set_facecolor(BOOK_STYLE["background"])
ax.set_facecolor(BOOK_STYLE["background"])

bars = ax.barh(
    pdf_2_1["accord"].str.title(),
    pdf_2_1["prevalence_pct"],
    height=0.68,
    color=BOOK_STYLE["accent"],
    edgecolor="none",
    zorder=3
)
for bar, value in zip(bars, pdf_2_1["prevalence_pct"]):
    ax.text(
        value + 0.8, bar.get_y() + bar.get_height()/2,
        f"{value:.2f}%", va="center", ha="left",
        fontsize=BOOK_SIZES["value_pt"],
        fontfamily=BOOK_TYPOGRAPHY["body"],
        color=BOOK_STYLE["text"]
    )

ax.set_xlim(0, float(pdf_2_1["prevalence_pct"].max()) * 1.18)
ax.set_xlabel("")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=100))
ax.tick_params(axis="y", length=0, labelsize=BOOK_SIZES["category_pt"], pad=10)
ax.tick_params(axis="x", length=0, labelsize=BOOK_SIZES["footer_pt"], pad=8)
ax.grid(axis="x", color=BOOK_STYLE["grid"], linewidth=0.8, zorder=0)
for spine in ax.spines.values():
    spine.set_visible(False)
apply_book_bar_layout(fig_2_1, bottom=0.08, right=0.96)
add_book_figure_header(
    fig_2_1,
    "The Distribution of Accord Prevalence",
    f"Thirty most prevalent accords among {valid_accord_count_ch2:,} fragrances with valid accord data",
    section="FRAGRANCE INTELLIGENCE  /  CHAPTER 2"
)
show_book_matplotlib(fig_2_1)

## 2.7 Figure 2.2 — Accord Prevalence Across Gender Classifications

A heatmap compares within-category prevalence for the 25 most prevalent accords overall. A shared scale is used across all three categories; the underlying table contains the full set of accord labels.

In [0]:
# ============================================================
# FIGURE 2.2 — ACCORD PREVALENCE ACROSS GENDER CLASSIFICATIONS
# ============================================================

heatmap_data_2_2 = (
    accord_gender_prevalence_ch2
    .join(
        accord_prevalence_ch2.select(
            "accord",
            F.col("prevalence_pct").alias("overall_prevalence_pct")
        ),
        on="accord",
        how="left"
    )
    .orderBy(F.desc("overall_prevalence_pct"), F.asc("accord"))
    .select("accord", "gender", "prevalence_pct")
    .toPandas()
)
top_25_accords_2_2 = (
    accord_prevalence_ch2.orderBy(F.desc("prevalence_pct"))
    .limit(25).select("accord").toPandas()["accord"].tolist()
)
heatmap_pivot_2_2 = (
    heatmap_data_2_2[heatmap_data_2_2["accord"].isin(top_25_accords_2_2)]
    .pivot(index="accord", columns="gender", values="prevalence_pct")
    .reindex(index=top_25_accords_2_2[::-1], columns=["men", "unisex", "women"])
    .fillna(0)
)

fig_2_2, ax = plt.subplots(figsize=(10, 9), dpi=180)
fig_2_2.patch.set_facecolor(BOOK_STYLE["background"])
ax.set_facecolor(BOOK_STYLE["background"])
im = ax.imshow(
    heatmap_pivot_2_2.values,
    aspect="auto",
    cmap=plt.cm.get_cmap("YlOrBr"),
    vmin=0,
    vmax=max(80, float(np.nanmax(heatmap_pivot_2_2.values)))
)
ax.set_yticks(np.arange(len(heatmap_pivot_2_2.index)))
ax.set_yticklabels(heatmap_pivot_2_2.index.str.title(), fontsize=BOOK_SIZES["category_pt"])
ax.set_xticks(np.arange(3))
ax.set_xticklabels(["Men", "Unisex", "Women"], fontsize=BOOK_SIZES["category_pt"])
ax.tick_params(length=0, pad=8)
for i in range(heatmap_pivot_2_2.shape[0]):
    for j in range(heatmap_pivot_2_2.shape[1]):
        value = heatmap_pivot_2_2.iloc[i, j]
        ax.text(j, i, f"{value:.1f}%", ha="center", va="center",
                fontsize=10, color=BOOK_STYLE["text"])
for spine in ax.spines.values():
    spine.set_visible(False)
cbar = fig_2_2.colorbar(im, ax=ax, fraction=0.035, pad=0.025)
cbar.set_label("Within-category prevalence (%)", fontsize=BOOK_SIZES["footer_pt"], color=BOOK_STYLE["muted"])
cbar.outline.set_visible(False)
cbar.ax.tick_params(labelsize=BOOK_SIZES["footer_pt"], length=0, colors=BOOK_STYLE["muted"])
fig_2_2.subplots_adjust(left=0.23, right=0.91, top=1 - BOOK_FIGURE_CHART_TOP_IN / fig_2_2.get_figheight(), bottom=0.08)
add_book_figure_header(
    fig_2_2,
    "Accord Prevalence Across Gender Classifications",
    "Within-category prevalence for the 25 most prevalent accords overall",
    section="FRAGRANCE INTELLIGENCE  /  CHAPTER 2"
)
show_book_matplotlib(fig_2_2)

## 2.8 Figure 2.3 — Gender-Specific Accord Prevalence Gaps

The chart ranks common accords by the difference between their highest and lowest prevalence across Men, Unisex and Women. Only accords with at least 10% overall prevalence are included.

In [0]:
# ============================================================
# FIGURE 2.3 — GENDER-SPECIFIC ACCORD PREVALENCE GAPS
# ============================================================

pdf_2_3 = (
    common_accord_gaps_ch2
    .orderBy(F.desc("gap_pp"), F.asc("accord"))
    .toPandas()
    .sort_values("gap_pp", ascending=True)
)

fig_2_3, ax = plt.subplots(figsize=(12, 9), dpi=180)
fig_2_3.patch.set_facecolor(BOOK_STYLE["background"])
ax.set_facecolor(BOOK_STYLE["background"])
bars = ax.barh(
    pdf_2_3["accord"].str.title(),
    pdf_2_3["gap_pp"],
    height=0.66,
    color=BOOK_STYLE["accent"],
    edgecolor="none",
    zorder=3
)
for bar, value in zip(bars, pdf_2_3["gap_pp"]):
    ax.text(value + 0.7, bar.get_y() + bar.get_height()/2,
            f"{value:.2f} pp", va="center", ha="left",
            fontsize=BOOK_SIZES["value_pt"], color=BOOK_STYLE["text"])
ax.set_xlim(0, float(pdf_2_3["gap_pp"].max()) * 1.2)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="y", length=0, labelsize=BOOK_SIZES["category_pt"], pad=10)
ax.tick_params(axis="x", length=0, labelsize=BOOK_SIZES["footer_pt"], pad=8)
ax.grid(axis="x", color=BOOK_STYLE["grid"], linewidth=0.8, zorder=0)
for spine in ax.spines.values():
    spine.set_visible(False)
apply_book_bar_layout(fig_2_3, bottom=0.08, right=0.96)
add_book_figure_header(
    fig_2_3,
    "Gender-Specific Accord Prevalence Gaps",
    "Highest minus lowest within-category prevalence; accords with at least 10% overall prevalence",
    section="FRAGRANCE INTELLIGENCE  /  CHAPTER 2"
)
show_book_matplotlib(fig_2_3)

## 2.9 Figure 2.4 — Accord Rating Differences from Gender-Specific Baselines

The diverging dot plot shows the highest and lowest rating differences within each gender category. The zero line represents the relevant category average. Only accord/category groups with at least 100 rated fragrances are included.

In [0]:
# ============================================================
# FIGURE 2.4 — ACCORD RATING DIFFERENCES FROM BASELINES
# ============================================================

pdf_rating_2_4 = accord_rating_analysis_ch2.toPandas()
gender_order_2_4 = ["men", "unisex", "women"]
extremes_2_4 = []
for gender in gender_order_2_4:
    group = pdf_rating_2_4[pdf_rating_2_4["gender"] == gender].copy()
    if not group.empty:
        extremes_2_4.append(group.nlargest(10, "rating_difference"))
        extremes_2_4.append(group.nsmallest(10, "rating_difference"))
pdf_2_4 = pd.concat(extremes_2_4, ignore_index=True).drop_duplicates(["gender", "accord"])
pdf_2_4["category"] = pdf_2_4["gender"].str.title()
pdf_2_4["label"] = pdf_2_4["accord"].str.title()

fig_2_4, axes = plt.subplots(1, 3, figsize=(16, 11), dpi=180, sharex=True)
fig_2_4.patch.set_facecolor(BOOK_STYLE["background"])
xmax_2_4 = max(abs(pdf_2_4["rating_difference"].min()), abs(pdf_2_4["rating_difference"].max())) * 1.22
for ax, gender in zip(axes, gender_order_2_4):
    ax.set_facecolor(BOOK_STYLE["background"])
    g = pdf_2_4[pdf_2_4["gender"] == gender].sort_values("rating_difference")
    y = np.arange(len(g))
    colors = [
        BOOK_STYLE["accent"] if value >= 0 else BOOK_STYLE["muted"]
        for value in g["rating_difference"]
    ]
    ax.hlines(y, 0, g["rating_difference"], color=BOOK_STYLE["rule"], linewidth=1.2, zorder=1)
    ax.scatter(g["rating_difference"], y, s=38, c=colors, zorder=3)
    ax.axvline(0, color=BOOK_STYLE["text"], linewidth=0.9, alpha=0.7)
    ax.set_yticks(y)
    ax.set_yticklabels(g["label"], fontsize=9)
    ax.set_title(gender.title(), loc="left", fontfamily=BOOK_TYPOGRAPHY["display"],
                 fontsize=15, color=BOOK_STYLE["text"], pad=10)
    ax.set_xlim(-xmax_2_4, xmax_2_4)
    ax.tick_params(axis="y", length=0, pad=5)
    ax.tick_params(axis="x", length=0, labelsize=9, pad=7)
    ax.grid(axis="x", color=BOOK_STYLE["grid"], linewidth=0.7, zorder=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
fig_2_4.supxlabel("Difference from gender-category average rating", y=0.045,
                  fontsize=BOOK_SIZES["footer_pt"], color=BOOK_STYLE["muted"])
fig_2_4.subplots_adjust(left=0.16, right=0.98, top=1 - BOOK_FIGURE_CHART_TOP_IN / fig_2_4.get_figheight(), bottom=0.11, wspace=0.55)
add_book_figure_header(
    fig_2_4,
    "Accord Rating Differences from Gender-Specific Baselines",
    "Ten highest and ten lowest differences per category; minimum 100 rated fragrances per group",
    section="FRAGRANCE INTELLIGENCE  /  CHAPTER 2"
)
show_book_matplotlib(fig_2_4)

## 2.10 Figure 2.5 — Rating Differences and Supporting Evidence

The scatter plot relates the number of rated fragrances in each accord/category group to its rating difference from the category baseline. Bubble size represents the total number of community ratings recorded for those fragrances. These counts measure platform activity, not sales or unique consumers.

In [0]:
# ============================================================
# FIGURE 2.5 — RATING DIFFERENCES AND SUPPORTING EVIDENCE
# ============================================================

pdf_2_5 = accord_rating_analysis_ch2.toPandas().copy()
pdf_2_5["gender_label"] = pdf_2_5["gender"].str.title()
fig_2_5, axes = plt.subplots(1, 3, figsize=(16, 7), dpi=180, sharey=True)
fig_2_5.patch.set_facecolor(BOOK_STYLE["background"])
gender_colors_2_5 = {
    "men": BOOK_STYLE["men"],
    "unisex": BOOK_STYLE["unisex"],
    "women": BOOK_STYLE["women"]
}
for ax, gender in zip(axes, gender_order_2_4):
    ax.set_facecolor(BOOK_STYLE["background"])
    g = pdf_2_5[pdf_2_5["gender"] == gender].copy()
    votes = g["total_rating_count"].fillna(0).clip(lower=0)
    sizes = 24 + 150 * np.sqrt(votes / max(float(votes.max()), 1))
    ax.scatter(
        g["rated_fragrance_count"],
        g["rating_difference"],
        s=sizes,
        color=gender_colors_2_5[gender],
        alpha=0.68,
        edgecolors=BOOK_STYLE["background"],
        linewidths=0.7,
        zorder=3
    )
    ax.axhline(0, color=BOOK_STYLE["text"], linewidth=0.9, alpha=0.7)
    ax.set_title(gender.title(), loc="left", fontfamily=BOOK_TYPOGRAPHY["display"],
                 fontsize=15, color=BOOK_STYLE["text"], pad=10)
    ax.set_xlabel("Rated fragrances", fontsize=BOOK_SIZES["footer_pt"], color=BOOK_STYLE["muted"])
    ax.tick_params(axis="both", length=0, labelsize=9, pad=7)
    ax.grid(axis="both", color=BOOK_STYLE["grid"], linewidth=0.7, zorder=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
axes[0].set_ylabel("Rating difference from category baseline", fontsize=BOOK_SIZES["footer_pt"],
                   color=BOOK_STYLE["muted"])
fig_2_5.subplots_adjust(left=0.09, right=0.98, top=1 - BOOK_FIGURE_CHART_TOP_IN / fig_2_5.get_figheight(), bottom=0.13, wspace=0.22)
add_book_figure_header(
    fig_2_5,
    "Rating Differences and Supporting Evidence",
    "X: rated fragrances  |  Y: difference from category baseline  |  Bubble size: total community ratings",
    section="FRAGRANCE INTELLIGENCE  /  CHAPTER 2"
)
show_book_matplotlib(fig_2_5)

## 2.11 Chapter 2 — Validation checks

Run these checks after the analytical preparation cell. They verify the known dataset scope and key published values, while also checking the gender-specific rating threshold and the prevalence-gap definition.

In [0]:
# ============================================================
# CHAPTER 2 — VALIDATION CHECKS
# ============================================================

assert valid_accord_count_ch2 == 68_511, (
    f"Unexpected valid-accord denominator: {valid_accord_count_ch2:,}"
)
assert distinct_accord_count_ch2 == 90, (
    f"Expected 90 normalized accord labels, found {distinct_accord_count_ch2}"
)

# Published category-level rating baselines.
baseline_check = {
    row["gender"]: round(float(row["gender_avg_rating"]), 3)
    for row in gender_rating_baseline_ch2.collect()
}
expected_baselines = {"men": 3.975, "unisex": 4.004, "women": 3.949}
for gender, expected in expected_baselines.items():
    assert baseline_check.get(gender) == expected, (
        f"{gender} baseline changed: expected {expected}, got {baseline_check.get(gender)}"
    )

# The rating comparison must respect the minimum group-size rule.
min_group = accord_rating_analysis_ch2.agg(
    F.min("rated_fragrance_count").alias("minimum")
).first()["minimum"]
assert min_group is None or min_group >= MIN_RATED_FRAGRANCES

# Gap must equal maximum minus minimum prevalence across the three categories.
gap_check = accord_gender_gaps_ch2.withColumn(
    "recalculated_gap",
    F.round(F.greatest("men", "unisex", "women") - F.least("men", "unisex", "women"), 2)
).filter(F.col("gap_pp") != F.col("recalculated_gap")).count()
assert gap_check == 0, f"{gap_check} prevalence-gap rows failed reconciliation."

print("Chapter 2 validation checks passed.")
print(f"Valid-accord denominator: {valid_accord_count_ch2:,}")
print(f"Normalized accord labels: {distinct_accord_count_ch2}")
print(f"Eligible accord/category rating groups: {accord_rating_analysis_ch2.count():,}")
print(f"Gender-specific rating baselines: {baseline_check}")

## 2.12 Optional Chapter 2 asset export

After reviewing the rendered prototypes, set `EXPORT_CHAPTER_2_ASSETS = True` to export the five figures and five styled tables. Export remains separate from visual review.

In [0]:
# ============================================================
# OPTIONAL CHAPTER 2 EXPORT — RUN AFTER VISUAL REVIEW
# ============================================================

from pathlib import Path

EXPORT_CHAPTER_2_ASSETS = False
CHAPTER_2_EXPORT_DIR = Path("/dbfs/FileStore/fragrance_book/chapter_2")

if EXPORT_CHAPTER_2_ASSETS:
    CHAPTER_2_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

    for figure, stem in [
        (fig_2_1, "figure_2_1_accord_prevalence"),
        (fig_2_2, "figure_2_2_gender_prevalence_heatmap"),
        (fig_2_3, "figure_2_3_gender_prevalence_gaps"),
        (fig_2_4, "figure_2_4_rating_differences"),
        (fig_2_5, "figure_2_5_rating_evidence")
    ]:
        figure.savefig(
            CHAPTER_2_EXPORT_DIR / f"{stem}.png",
            dpi=400,
            bbox_inches=None,
            facecolor=BOOK_STYLE["background"]
        )
        figure.savefig(
            CHAPTER_2_EXPORT_DIR / f"{stem}.svg",
            bbox_inches=None,
            facecolor=BOOK_STYLE["background"]
        )

    for table_name, table_object in {
        "table_2_1_overall_accord_prevalence": gt_table_2_1,
        "table_2_2_gender_accord_prevalence": gt_table_2_2,
        "table_2_3_gender_prevalence_gaps": gt_table_2_3,
        "table_2_4_gender_rating_baselines": gt_table_2_4,
        "table_2_5_accord_ratings_engagement": gt_table_2_5
    }.items():
        table_object.write_raw_html(
            str(CHAPTER_2_EXPORT_DIR / f"{table_name}.html")
        )

    print(f"Chapter 2 assets exported to: {CHAPTER_2_EXPORT_DIR}")
else:
    print("Chapter 2 export is off. Review the rendered prototypes first.")